<a href="https://colab.research.google.com/github/VictorVega03/Programacion-de-sistemas-base-2/blob/main/Copia_de_PSSB_II_Colab_U2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UNIVERSIDAD AUTÓNOMA DE TAMAULIPAS
# Programación de Sistemas de Base II - Algoritmo del Número de Valor (Value-Numbering) para Construcción de DAGs
**Asignatura:** Programación de Sistemas de Base II (9.º Semestre)  
**Unidad II:** Generación de código intermedio  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---

## 1. Objetivos de Aprendizaje
1. Implementar el Algoritmo del Número de Valor (Value-Number Method) para convertir un Árbol Sintáctico Abstracto (AST) con expresiones repetidas en un Grafo Dirigido Acíclico (DAG) comprimido.  
2. Implementar las estructuras internas de Código de Tres Direcciones (TAC), generando y comparando dinámicamente tablas de Cuádruplas, Triplos y Triplos Indirectos para la expresión fuente `a = -b * (c + d)`.  
3. Implementar un Motor Ejecutor / Máquina Virtual de Pila (Stack VM) que traduce una expresión en C/Java a Bytecode basado en RPN y ejecuta el ciclo Fetch-Decode-Execute visualizando el estado de la pila en tiempo real.
----

## 2. Sección Práctica 1:

### [CÓDIGO 1.1] Algoritmo Value-Number


In [15]:
from typing import Dict, Tuple, Union, Optional

# ------------------------------------------------------------------------------
# 1. DEFINICIÓN DE ESTRUCTURAS DE DATOS PARA DAG Y AST
# ------------------------------------------------------------------------------

class DAGNode:
    """Representa un nodo único dentro del Grafo Dirigido Acíclico (DAG)."""
    def __init__(self, value_number: int, op: str,
                 left: Optional[int] = None, right: Optional[int] = None):
        self.value_number = value_number  # Índice entero único (# de Valor)
        self.op = op                      # Operador o tipo de hoja ("ID", "CONST", "+", "*", "-")
        self.left = left                  # # de valor del hijo izquierdo
        self.right = right                # # de valor del hijo derecho
        self.parents_count = 0            # Contador de referencias/padres para métricas

    def is_leaf(self) -> bool:
        return self.left is None and self.right is None

    def signature(self) -> Tuple:
        """Devuelve la tupla de firma única utilizada para búsquedas en la Tabla Hash."""
        return (self.op, self.left, self.right)

    def __repr__(self):
        if self.is_leaf():
            return f"Node #{self.value_number}: Leaf({self.op})"
        return f"Node #{self.value_number}: Op('{self.op}', Left=# {self.left}, Right=# {self.right})"

# ------------------------------------------------------------------------------
# 2. ALGORITMO DEL NÚMERO DE VALOR (VALUE-NUMBERING BUILDER)
# ------------------------------------------------------------------------------

class DAGBuilderValueNumbering:
    """
    Implementa el Algoritmo 6.3 de Aho et al. (Dragon Book) mediante
    una Tabla Hash de firmas para asegurar la compartición de nodos.
    """
    def __init__(self):
        self.nodes: Dict[int, DAGNode] = {}              # Mapa: Value_Number -> DAGNode
        self.hash_table: Dict[Tuple, int] = {}           # Tabla Hash: Signature -> Value_Number
        self.next_value_number = 1

    def get_or_create_leaf(self, leaf_type: str, lexeme: str) -> int:
        """Busca o crea un nodo hoja (ID o CONST) garantizando unicidad."""
        sig = (leaf_type, lexeme, None)
        if sig in self.hash_table:
            vn = self.hash_table[sig]
            self.nodes[vn].parents_count += 1
            return vn

        vn = self.next_value_number
        self.next_value_number += 1
        node = DAGNode(vn, f"{leaf_type}:{lexeme}")
        node.parents_count = 1

        self.nodes[vn] = node
        self.hash_table[sig] = vn
        return vn

    def get_or_create_op(self, op: str, left_vn: int, right_vn: int) -> int:
        """
        Busca o crea un nodo operador binario. Si la firma <op, izq, der>
        ya existe, reutiliza el número de valor previo (eliminando redundancia).
        """
        sig = (op, left_vn, right_vn)
        if sig in self.hash_table:
            vn = self.hash_table[sig]
            self.nodes[vn].parents_count += 1
            print(f"   [REUTILIZACIÓN DETECTADA] Firma {sig} ya existe -> Retorna Nodo #{vn}")
            return vn

        vn = self.next_value_number
        self.next_value_number += 1
        node = DAGNode(vn, op, left_vn, right_vn)
        node.parents_count = 1

        # Incrementar referencias en los hijos
        self.nodes[left_vn].parents_count += 1
        self.nodes[right_vn].parents_count += 1

        self.nodes[vn] = node
        self.hash_table[sig] = vn
        print(f"   [CREACIÓN DE NODO] Nueva Firma {sig} -> Asignado Nodo #{vn}")
        return vn

    def print_value_number_table(self):
        """Imprime la tabla completa de números de valor con la estructura del DAG."""
        print("\n=================================================================")
        print("    TABLA DE NÚMEROS DE VALOR (ESTRUCTURA DEL DAG COMPRIMIDO)")
        print("=================================================================")
        print(f"{'# Valor':<10} | {'Firma / Operación':<30} | {'Referencias (Padres)'}")
        print("-" * 65)
        for vn in sorted(self.nodes.keys()):
            node = self.nodes[vn]
            if node.is_leaf():
                desc = f"HOJA -> {node.op}"
            else:
                desc = f"OP('{node.op}') [Izq: #{node.left}, Der: #{node.right}]"
            print(f" #{vn:<8} | {desc:<30} | {node.parents_count}")
        print("=================================================================\n")

# ------------------------------------------------------------------------------
# 3. RECORRIDO DE ÁRBOLES Y CONVERSIÓN DE AST A DAG
# ------------------------------------------------------------------------------

class ASTSimpleNode:
    """Clase para representar un AST convencional (sin compartir nodos)."""
    def __init__(self, op: str, left=None, right=None):
        self.op = op
        self.left = left
        self.right = right

    def count_nodes(self) -> int:
        """Cuenta recursivamente el total de nodos en un AST no optimizado."""
        count = 1
        if self.left:
            count += self.left.count_nodes()
        if self.right:
            count += self.right.count_nodes()
        return count

def convert_ast_to_dag(ast_node: ASTSimpleNode, builder: DAGBuilderValueNumbering) -> int:
    """
    Recorre el AST en postorden y utiliza el DAGBuilder para
    construir el DAG comprimido equivalente.
    """
    if ast_node.left is None and ast_node.right is None:
        # Es una hoja: determinar si es ID o constante numérica
        if ast_node.op.isdigit():
            return builder.get_or_create_leaf("CONST", ast_node.op)
        else:
            return builder.get_or_create_leaf("ID", ast_node.op)

    # Recorrido Postorden: Primero procesar subárbol izquierdo y derecho
    left_vn = convert_ast_to_dag(ast_node.left, builder)
    right_vn = convert_ast_to_dag(ast_node.right, builder)

    # Procesar la raíz del subárbol actual
    return builder.get_or_create_op(ast_node.op, left_vn, right_vn)

# ------------------------------------------------------------------------------
# 4. EJECUCIÓN Y DEMOSTRACIÓN DEL LABORATORIO EN COLAB
# ------------------------------------------------------------------------------

def run_dag_lab_demo():
    print("=================================================================")
    print("   LABORATORIO COLAB: CONSTRUCCIÓN DE DAGs MEDIANTE VALUE-NUMBERING")
    print("   Expresión fuente: (a + b) * (a + b)")
    print("=================================================================\n")

    # Construcción manual del AST unmitigated (con nodos duplicados)
    # Subexpresión izquierda: (a + b)
    ast_left_sum = ASTSimpleNode("+", ASTSimpleNode("a"), ASTSimpleNode("b"))
    # Subexpresión derecha: (a + b) [Duplicada]
    ast_right_sum = ASTSimpleNode("+", ASTSimpleNode("a"), ASTSimpleNode("b"))
    # Raíz: multiplicación
    ast_root = ASTSimpleNode("*", ast_left_sum, ast_right_sum)

    total_ast_nodes = ast_root.count_nodes()
    print(f"1. Conteo de Nodos en AST Convencional (Sin compartir): {total_ast_nodes} nodos")
    print("   Estructura: [a, b, +, a, b, +, *]\n")

    print("2. Ejecutando Algoritmo de Value-Numbering en Postorden...")
    builder = DAGBuilderValueNumbering()
    root_vn = convert_ast_to_dag(ast_root, builder)

    # Mostrar la tabla de resultados
    builder.print_value_number_table()

    total_dag_nodes = len(builder.nodes)
    savings = ((total_ast_nodes - total_dag_nodes) / total_ast_nodes) * 100

    print("3. MÉTRICAS DE COMPRESIÓN Y EFICIENCIA:")
    print(f"   - Nodos en AST Original : {total_ast_nodes}")
    print(f"   - Nodos en DAG Resultante: {total_dag_nodes}")
    print(f"   - Reducción de Nodos     : {total_ast_nodes - total_dag_nodes} nodos ahorrados ({savings:.1f}%)")
    print(f"   - Raíz del DAG           : Nodo #{root_vn}")
    print("=================================================================")

# Ejecutar el laboratorio
if __name__ == "__main__":
    run_dag_lab_demo()

   LABORATORIO COLAB: CONSTRUCCIÓN DE DAGs MEDIANTE VALUE-NUMBERING
   Expresión fuente: (a + b) * (a + b)

1. Conteo de Nodos en AST Convencional (Sin compartir): 7 nodos
   Estructura: [a, b, +, a, b, +, *]

2. Ejecutando Algoritmo de Value-Numbering en Postorden...
   [CREACIÓN DE NODO] Nueva Firma ('+', 1, 2) -> Asignado Nodo #3
   [REUTILIZACIÓN DETECTADA] Firma ('+', 1, 2) ya existe -> Retorna Nodo #3
   [CREACIÓN DE NODO] Nueva Firma ('*', 3, 3) -> Asignado Nodo #4

    TABLA DE NÚMEROS DE VALOR (ESTRUCTURA DEL DAG COMPRIMIDO)
# Valor    | Firma / Operación              | Referencias (Padres)
-----------------------------------------------------------------
 #1        | HOJA -> ID:a                   | 3
 #2        | HOJA -> ID:b                   | 3
 #3        | OP('+') [Izq: #1, Der: #2]     | 4
 #4        | OP('*') [Izq: #3, Der: #3]     | 1

3. MÉTRICAS DE COMPRESIÓN Y EFICIENCIA:
   - Nodos en AST Original : 7
   - Nodos en DAG Resultante: 4
   - Reducción de Nodos     : 3

## 3. Sección Práctica 2:

### [CÓDIGO 2.1] Estructuras Internas

In [14]:
from typing import List, Tuple, Union, Optional

# ------------------------------------------------------------------------------
# 1. REPRESENTACIÓN DE ESTRUCTURAS INTERNAS (CUÁDRUPLAS, TRIPLOS E INDIRECTOS)
# ------------------------------------------------------------------------------

class Quadruple:
    """Estructura de registro de 4 campos: (op, arg1, arg2, result)"""
    def __init__(self, op: str, arg1: str, arg2: Optional[str], result: str):
        self.op = op
        self.arg1 = arg1
        self.arg2 = arg2 if arg2 is not None else "-"
        self.result = result

    def __repr__(self):
        return f"({self.op:<8}, {self.arg1:<6}, {self.arg2:<6}, {self.result:<6})"

class Triple:
    """Estructura de registro de 3 campos: (op, arg1, arg2)"""
    def __init__(self, op: str, arg1: str, arg2: Optional[str]):
        self.op = op
        self.arg1 = arg1
        self.arg2 = arg2 if arg2 is not None else "-"

    def __repr__(self):
        return f"({self.op:<8}, {self.arg1:<8}, {self.arg2:<8})"

# ------------------------------------------------------------------------------
# 2. GENERADOR DE CÓDIGO INTERMEDIO (TAC GENERATOR)
# ------------------------------------------------------------------------------

class TACCodeGenerator:
    """
    Toma una secuencia de operaciones sintácticas y genera simultáneamente
    sus representaciones internas en Cuádruplas, Triplos y Triplos Indirectos.
    """
    def __init__(self):
        self.quads: List[Quadruple] = []
        self.triples: List[Triple] = []
        self.indirect_pointers: List[int] = []  # Vector de punteros para Triplos Indirectos
        self.temp_counter = 0

    def new_temp(self) -> str:
        self.temp_counter += 1
        return f"t{self.temp_counter}"

    def emit_quad(self, op: str, arg1: str, arg2: Optional[str], result: str):
        quad = Quadruple(op, arg1, arg2, result)
        self.quads.append(quad)

    def emit_triple(self, op: str, arg1: str, arg2: Optional[str]) -> str:
        idx = len(self.triples)
        triple = Triple(op, arg1, arg2)
        self.triples.append(triple)
        self.indirect_pointers.append(idx)
        return f"({idx})"  # Retorna la referencia entre paréntesis (i)

    def print_quadruples_table(self):
        print("\n=================================================================")
        print("                TABLA DE CUÁDRUPLAS (QUADRUPLES)")
        print("=================================================================")
        print(f"{'Índice':<8} | {'op':<8} | {'arg1':<6} | {'arg2':<6} | {'resultado':<8}")
        print("-" * 65)
        for idx, q in enumerate(self.quads):
            print(f" ({idx})    | {q.op:<8} | {q.arg1:<6} | {q.arg2:<6} | {q.result:<8}")
        print("=================================================================\n")

    def print_triples_table(self):
        print("=================================================================")
        print("                  TABLA DE TRIPLOS (TRIPLES)")
        print("=================================================================")
        print(f"{'Índice':<8} | {'op':<8} | {'arg1':<8} | {'arg2':<8}")
        print("-" * 65)
        for idx, t in enumerate(self.triples):
            print(f" ({idx})    | {t.op:<8} | {t.arg1:<8} | {t.arg2:<8}")
        print("=================================================================\n")

    def print_indirect_triples_table(self):
        print("=================================================================")
        print("        TABLA DE TRIPLOS INDIRECTOS (INDIRECT TRIPLES)")
        print("=================================================================")
        print(f"{'Orden Execution':<18} | {'Puntero a Triplo':<20} | {'Contenido del Triplo'}")
        print("-" * 65)
        for exec_order, triple_idx in enumerate(self.indirect_pointers):
            t = self.triples[triple_idx]
            print(f"  Instrucción [{exec_order}]   --->   Puntero a Triplo ({triple_idx}) | {t}")
        print("=================================================================\n")

# ------------------------------------------------------------------------------
# 3. DEMOSTRACIÓN DEL LABORATORIO: COMPILANDO `a = -b * (c + d)`
# ------------------------------------------------------------------------------

def run_tac_lab_demo():
    print("=================================================================")
    print("    LABORATORIO COLAB: GENERACIÓN DE ESTRUCTURAS TAC (MÓDULO 2.2.1)")
    print("    Expresión Fuente C/Java: a = -b * (c + d)")
    print("=================================================================\n")

    gen = TACCodeGenerator()

    # Paso 1: Menos Unario a 'b' -> t1 = -b / Triplo (0)
    t1 = gen.new_temp()
    gen.emit_quad("uminus", "b", None, t1)
    r0 = gen.emit_triple("uminus", "b", None)

    # Paso 2: Suma de 'c' + 'd' -> t2 = c + d / Triplo (1)
    t2 = gen.new_temp()
    gen.emit_quad("+", "c", "d", t2)
    r1 = gen.emit_triple("+", "c", "d")

    # Paso 3: Multiplicación de los resultados parciales -> t3 = t1 * t2 / Triplo (2)
    t3 = gen.new_temp()
    gen.emit_quad("*", t1, t2, t3)
    r2 = gen.emit_triple("*", r0, r1)

    # Paso 4: Asignación a la variable 'a' -> a = t3 / Triplo (3)
    gen.emit_quad("=", t3, None, "a")
    gen.emit_triple("=", r2, "a")

    # Imprimir tablas resultantes
    gen.print_quadruples_table()
    gen.print_triples_table()
    gen.print_indirect_triples_table()

    # Simulación de Reordenamiento en Triplos Indirectos (Optimización)
    print(">>> SIMULACIÓN DE OPTIMIZACIÓN EN TRIPLOS INDIRECTOS:")
    print("    Si se permuta el orden de ejecución sin alterar la tabla física de triplos:")
    # Intercambiamos orden de ejecución por ejemplo: ejecutar suma primero
    gen.indirect_pointers = [1, 0, 2, 3]
    print("    Nuevo orden de ejecución en arreglo 'Instrucciones': [1, 0, 2, 3]")
    gen.print_indirect_triples_table()

# Ejecutar la simulación
if __name__ == "__main__":
    run_tac_lab_demo()

    LABORATORIO COLAB: GENERACIÓN DE ESTRUCTURAS TAC (MÓDULO 2.2.1)
    Expresión Fuente C/Java: a = -b * (c + d)


                TABLA DE CUÁDRUPLAS (QUADRUPLES)
Índice   | op       | arg1   | arg2   | resultado
-----------------------------------------------------------------
 (0)    | uminus   | b      | -      | t1      
 (1)    | +        | c      | d      | t2      
 (2)    | *        | t1     | t2     | t3      
 (3)    | =        | t3     | -      | a       

                  TABLA DE TRIPLOS (TRIPLES)
Índice   | op       | arg1     | arg2    
-----------------------------------------------------------------
 (0)    | uminus   | b        | -       
 (1)    | +        | c        | d       
 (2)    | *        | (0)      | (1)     
 (3)    | =        | (2)      | a       

        TABLA DE TRIPLOS INDIRECTOS (INDIRECT TRIPLES)
Orden Execution    | Puntero a Triplo     | Contenido del Triplo
-----------------------------------------------------------------
  Instrucción [0]   --

## 4. Sección Práctica 3:

### [CÓDIGO 3.1] Máquina Virtual de Pila.


In [13]:
import enum
from typing import List, Dict, Union, Any

# ------------------------------------------------------------------------------
# 1. DEFINICIÓN DEL REPERTORIO DE INSTRUCCIONES (BYTECODE OPCODES)
# ------------------------------------------------------------------------------

class OpCode(enum.Enum):
    PUSH_CONST = "PUSH_CONST"  # Empuja una constante numérica a la pila
    PUSH_VAR   = "PUSH_VAR"    # Empuja el valor de una variable a la pila
    POP_VAR    = "POP_VAR"     # Extrae la cima y la guarda en una variable
    ADD        = "ADD"         # Extrae 2 operandos, suma y empuja resultado
    SUB        = "SUB"         # Extrae 2 operandos, resta (izq - der) y empuja
    MULT       = "MULT"        # Extrae 2 operandos, multiplica y empuja
    DIV        = "DIV"         # Extrae 2 operandos, divide (izq / der) y empuja
    DUP        = "DUP"         # Duplica la cima de la pila
    SWAP       = "SWAP"        # Intercambia los dos elementos superiores
    PRINT      = "PRINT"       # Muestra y extrae el valor en la cima
    HALT       = "HALT"        # Detiene la ejecución de la VM

class Instruction:
    """Representa una instrucción binaria de Bytecode (OpCode + Operando Opcional)."""
    def __init__(self, opcode: OpCode, operand: Any = None):
        self.opcode = opcode
        self.operand = operand

    def __repr__(self):
        if self.operand is not None:
            return f"{self.opcode.value:<12} {self.operand}"
        return f"{self.opcode.value:<12}"

# ------------------------------------------------------------------------------
# 2. MOTOR DE LA MÁQUINA VIRTUAL DE PILA (STACK VM)
# ------------------------------------------------------------------------------

class StackVirtualMachine:
    """
    Simula el comportamiento de un entorno de ejecución basado en pila
    con memoria de variables, pila de operandos y puntero de instrucciones (IP).
    """
    def __init__(self):
        self.operand_stack: List[Union[int, float]] = []  # Pila de operandos LIFO
        self.memory: Dict[str, Union[int, float]] = {}    # Memoria principal de variables
        self.code: List[Instruction] = []                 # Arreglo lineal de bytecode
        self.ip: int = 0                                  # Instruction Pointer

    def load_code(self, code: List[Instruction]):
        self.code = code
        self.ip = 0
        self.operand_stack.clear()

    def step(self) -> bool:
        """Ejecuta un ciclo Fetch-Decode-Execute. Devuelve False al encontrar HALT."""
        if self.ip >= len(self.code):
            return False

        inst = self.code[self.ip]
        self.ip += 1

        # FETCH & DECODE
        op = inst.opcode
        arg = inst.operand

        # EXECUTE
        if op == OpCode.PUSH_CONST:
            self.operand_stack.append(arg)

        elif op == OpCode.PUSH_VAR:
            if arg not in self.memory:
                raise NameError(f"Error en tiempo de ejecución: Variable '{arg}' no inicializada.")
            self.operand_stack.append(self.memory[arg])

        elif op == OpCode.POP_VAR:
            if not self.operand_stack:
                raise IndexError("Stack Underflow al ejecutar POP_VAR.")
            val = self.operand_stack.pop()
            self.memory[arg] = val

        elif op in (OpCode.ADD, OpCode.SUB, OpCode.MULT, OpCode.DIV):
            if len(self.operand_stack) < 2:
                raise IndexError(f"Stack Underflow al ejecutar operación aritmética {op.value}.")
            b = self.operand_stack.pop()  # Operando derecho (cima)
            a = self.operand_stack.pop()  # Operando izquierdo

            if op == OpCode.ADD:
                res = a + b
            elif op == OpCode.SUB:
                res = a - b
            elif op == OpCode.MULT:
                res = a * b
            elif op == OpCode.DIV:
                if b == 0:
                    raise ZeroDivisionError("División por cero en tiempo de ejecución de la VM.")
                res = a / b
            self.operand_stack.append(res)

        elif op == OpCode.DUP:
            if not self.operand_stack:
                raise IndexError("Stack Underflow al ejecutar DUP.")
            self.operand_stack.append(self.operand_stack[-1])

        elif op == OpCode.SWAP:
            if len(self.operand_stack) < 2:
                raise IndexError("Stack Underflow al ejecutar SWAP.")
            self.operand_stack[-1], self.operand_stack[-2] = self.operand_stack[-2], self.operand_stack[-1]

        elif op == OpCode.PRINT:
            if not self.operand_stack:
                raise IndexError("Stack Underflow al ejecutar PRINT.")
            val = self.operand_stack.pop()
            print(f"    [SALIDA VM] => {val}")

        elif op == OpCode.HALT:
            return False

        return True

    def run_with_trace(self):
        """Ejecuta el programa completo mostrando la traza paso a paso."""
        print("\n=================================================================")
        print("         TRAZA DE EJECUCIÓN PASO A PASO EN LA MÁQUINA VIRTUAL")
        print("=================================================================")
        print(f"{'IP':<5} | {'Instrucción Emitida':<22} | {'Estado de la Pila (LIFO)'}")
        print("-" * 65)

        while self.ip < len(self.code):
            current_ip = self.ip
            inst_str = str(self.code[current_ip])
            running = self.step()
            stack_state = str(self.operand_stack)
            print(f" {current_ip:<4} | {inst_str:<22} | {stack_state}")
            if not running:
                break
        print("=================================================================\n")

# ------------------------------------------------------------------------------
# 3. DEMOSTRACIÓN DEL LABORATORIO: TRADUCCIÓN Y EJECUCIÓN DE EXPRESSIÓN
# ------------------------------------------------------------------------------

def run_bytecode_lab_demo():
    print("=================================================================")
    print("    LABORATORIO COLAB: COMPILACIÓN A BYTECODE Y EJECUCIÓN (MÓDULO 2.2.2)")
    print("    Expresión fuente C/Java: x = (a + b) * (c - d)")
    print("=================================================================")

    # Inicializar la Máquina Virtual
    vm = StackVirtualMachine()

    # Cargar valores iniciales de la memoria de variables (Entorno)
    vm.memory["a"] = 5
    vm.memory["b"] = 3
    vm.memory["c"] = 10
    vm.memory["d"] = 4

    print("\n1. ESTADO INICIAL DE LA MEMORIA DE VARIABLES:")
    for var, val in vm.memory.items():
        print(f"    Variable '{var}' = {val}")

    # Programa en Bytecode equivalente a: x = (a + b) * (c - d)
    # En RPN: a b + c d - * x =
    bytecode_program = [
        Instruction(OpCode.PUSH_VAR, "a"),  # PUSH a (5)
        Instruction(OpCode.PUSH_VAR, "b"),  # PUSH b (3)
        Instruction(OpCode.ADD),            # ADD (5 + 3 = 8)
        Instruction(OpCode.PUSH_VAR, "c"),  # PUSH c (10)
        Instruction(OpCode.PUSH_VAR, "d"),  # PUSH d (4)
        Instruction(OpCode.SUB),            # SUB (10 - 4 = 6)
        Instruction(OpCode.MULT),           # MULT (8 * 6 = 48)
        Instruction(OpCode.POP_VAR, "x"),   # POP_VAR x (Guarda 48 en 'x')
        Instruction(OpCode.PUSH_VAR, "x"),  # PUSH x para verificación de salida
        Instruction(OpCode.PRINT),          # PRINT muestra 48
        Instruction(OpCode.HALT)            # HALT
    ]

    # Cargar y ejecutar programa en la VM
    vm.load_code(bytecode_program)
    vm.run_with_trace()

    print("2. ESTADO FINAL DE LA MEMORIA TRAS LA EJECUCIÓN:")
    for var, val in vm.memory.items():
        print(f"    Variable '{var}' = {val}")
    print("=================================================================")

# Ejecutar el laboratorio
if __name__ == "__main__":
    run_bytecode_lab_demo()

    LABORATORIO COLAB: COMPILACIÓN A BYTECODE Y EJECUCIÓN (MÓDULO 2.2.2)
    Expresión fuente C/Java: x = (a + b) * (c - d)

1. ESTADO INICIAL DE LA MEMORIA DE VARIABLES:
    Variable 'a' = 5
    Variable 'b' = 3
    Variable 'c' = 10
    Variable 'd' = 4

         TRAZA DE EJECUCIÓN PASO A PASO EN LA MÁQUINA VIRTUAL
IP    | Instrucción Emitida    | Estado de la Pila (LIFO)
-----------------------------------------------------------------
 0    | PUSH_VAR     a         | [5]
 1    | PUSH_VAR     b         | [5, 3]
 2    | ADD                    | [8]
 3    | PUSH_VAR     c         | [8, 10]
 4    | PUSH_VAR     d         | [8, 10, 4]
 5    | SUB                    | [8, 6]
 6    | MULT                   | [48]
 7    | POP_VAR      x         | []
 8    | PUSH_VAR     x         | [48]
    [SALIDA VM] => 48
 9    | PRINT                  | []
 10   | HALT                   | []

2. ESTADO FINAL DE LA MEMORIA TRAS LA EJECUCIÓN:
    Variable 'a' = 5
    Variable 'b' = 3
    Variable 'c' = 1